In [1]:
import torch
import torch.nn as nn
import numpy as np
from DDBSCAN import Raster_DBSCAN
from torch.utils.data import Dataset,DataLoader
import torch.optim as optim
from Models import *
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.colors as mcolors
from ModelSocial import LaneOccupancySocialLSTM
from TrainSocialLSTM import extract_trajectories,prepare_training_data
# times new roman font
plt.rcParams["font.family"] = "Times New Roman"
seed = 414
# np.random.seed(seed)
colors = np.random.rand(600, 3)
colors = np.concatenate([np.array([[0,0,0]]),colors],axis = 0)
colormap = mcolors.ListedColormap(colors)

In [18]:
def prepare_numerical_data(traffic_context, batch_size=4):
    """
    Prepare numerical data from traffic context for the simplified model
    
    Args:
        traffic_context: Occupancy grid [batch, lane_cell_num, seq_length]
                         Each element is trajectory ID (0 = unoccupied)
        batch_size: Desired batch size
        
    Returns:
        inputs: Input tensor [batch_size, input_frames, 3]
        targets: Target positions [batch_size, output_size]
    """
    batch = traffic_context.size(0)
    lane_cells = traffic_context.size(1)
    seq_length = traffic_context.size(2)
    
    # Set parameters
    input_frames = 10
    window_size = input_frames + 1  # +1 for target
    
    inputs_list = []
    targets_list = []
    
    # Process each scenario in the batch
    for b in range(batch):
        # Find unique vehicle IDs
        unique_ids = torch.unique(traffic_context[b])
        unique_ids = unique_ids[unique_ids > 0]
        
        # Process each vehicle trajectory
        for vehicle_id in unique_ids:
            # Find positions of this vehicle at each time step
            positions = []
            for t in range(seq_length):
                # Find cells occupied by this vehicle
                cells = torch.nonzero(traffic_context[b, :, t] == vehicle_id, as_tuple=True)[0]
                if len(cells) > 0:
                    positions.append(cells[0].item())
                else:
                    positions.append(-1)
            
            # Create sliding windows
            for start_idx in range(seq_length - window_size + 1):
                window = positions[start_idx:start_idx + window_size]
                
                # Skip windows with missing positions
                if -1 in window:
                    continue
                
                # Extract input and target positions
                input_positions = window[:input_frames]
                target_position = window[input_frames]
                
                # Create input tensor with [position, front_dist, back_dist]
                input_data = torch.zeros(input_frames, 3)
        
                for t, pos in enumerate(input_positions):
                    # Set tracked vehicle position
                    input_data[t, 0] = pos

                    # Find front vehicle
                    front_dist = -999 
                    for front_pos in range(pos - 1, -1, -1):
                        curr_id = traffic_context[b, front_pos, start_idx + t]
                        if curr_id > 0 and curr_id != vehicle_id:
                            front_dist = front_pos - pos  # Will be negative
                            break
                    back_dist = 999
                    for back_pos in range(pos + 1, lane_cells):
                        curr_id = traffic_context[b, back_pos, start_idx + t]
                        if curr_id > 0 and curr_id != vehicle_id:
                            back_dist = back_pos - pos
                            break
                    input_data[t, 1] = front_dist 
                    input_data[t, 2] = back_dist

                
                inputs_list.append(input_data)
                targets_list.append(torch.tensor([target_position]))
                
                # If we have enough samples for a batch, yield them
                if len(inputs_list) >= batch_size:
                    inputs_batch = torch.stack(inputs_list[:batch_size])
                    targets_batch = torch.stack(targets_list[:batch_size])
                    
                    inputs_list = inputs_list[batch_size:]
                    targets_list = targets_list[batch_size:]
                    
                    yield inputs_batch, targets_batch
    
    # Yield any remaining samples
    if len(inputs_list) > 0:
        inputs_batch = torch.stack(inputs_list)
        targets_batch = torch.stack(targets_list)
        yield inputs_batch, targets_batch

In [28]:
out_folder = r"D:\TimeSpaceDiagramDataset\SocialLSTMDataset"
val_folder = os.path.join(out_folder,"val")
batch_size = 4
time_span = 100
val_dataset = TrajDataset(r"D:\TimeSpaceDiagramDataset\EncoderDecoder_EvenlySampled_FreeflowAug_0914_5res_lanechange_signal\100_frame\val",time_span)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
# find out the number of 1 over the total number of elements
counts = 0
for batch_idx,batch in enumerate(tqdm(val_loader)):
    traj_id = batch['traj_id']
    gen = prepare_numerical_data(traj_id)
    break

  0%|          | 0/61500 [00:02<?, ?it/s]


In [55]:
import h5py
out_folder = r"D:\TimeSpaceDiagramDataset\SocialLSTMDataset"
val_folder = os.path.join(out_folder,"val")
os.makedirs(val_folder,exist_ok=True)
train_folder = os.path.join(out_folder,"train")
os.makedirs(train_folder,exist_ok=True)
train_count = 40000
val_count = 7000
batch_size = 4
p = 0.2
time_span = 100
val_dataset = TrajDataset(r"D:\TimeSpaceDiagramDataset\EncoderDecoder_EvenlySampled_FreeflowAug_0914_5res_lanechange_signal\100_frame\val",time_span)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
# find out the number of 1 over the total number of elements
counts = 0
h5_file = h5py.File(os.path.join(val_folder,"val_dataset.h5"),'w')
for batch_idx,batch in enumerate(tqdm(val_loader)):
    traj_id = batch['traj_id']
    gen = prepare_numerical_data(traj_id)
    # Create a group for this batch
    batch_group = h5_file.create_group(f'batch_{batch_idx}')
    batch_count = 0
    while True:
        try:
            inputs, targets = next(gen)
            if np.random.rand() > p:
                continue
            # inputs: [batch_size, input_frames, 3 (pos, front_dist, back_dist)], targets: [batch_size, 1]
            mini_batch = batch_group.create_group(f'mini_batch_{batch_count}')
            mini_batch.create_dataset('inputs', data=inputs.numpy(), compression="gzip")
            mini_batch.create_dataset('targets', data=targets.numpy(), compression="gzip")
            counts += inputs.size(0)
            batch_count += 1
        except StopIteration:
            break
    batch_group.attrs['total_samples'] = batch_count
    if counts > val_count:
        break
h5_file.attrs['total_trajectory_samples'] = counts
h5_file.close()

train_dataset = TrajDataset(r"D:\TimeSpaceDiagramDataset\EncoderDecoder_EvenlySampled_FreeflowAug_0914_5res_lanechange_signal\100_frame\train",time_span)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
# find out the number of 1 over the total number of elements
counts = 0
h5_file = h5py.File(os.path.join(train_folder,"train_dataset.h5"),'w')
for batch_idx,batch in enumerate(tqdm(train_loader)):
    traj_id = batch['traj_id']
    gen = prepare_numerical_data(traj_id)
    # Create a group for this batch
    batch_group = h5_file.create_group(f'batch_{batch_idx}')
    batch_count = 0
    while True:
        try:
            inputs, targets = next(gen)
            if np.random.rand() > p:
                continue
            mini_batch = batch_group.create_group(f'mini_batch_{batch_count}')
            mini_batch.create_dataset('inputs', data=inputs.numpy(), compression="gzip")
            mini_batch.create_dataset('targets', data=targets.numpy(), compression="gzip")
            counts += inputs.size(0)
            batch_count += 1
        except StopIteration:
            break
    batch_group.attrs['total_samples'] = batch_count
    if counts > train_count:
        break
h5_file.attrs['total_trajectory_samples'] = counts
h5_file.close()



  0%|          | 187/246000 [24:02<526:36:13,  7.71s/it]


In [57]:
from Dataset import TrajectoryDataset

In [89]:
train_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\train\train_dataset.h5')
val_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\val\val_dataset.h5')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=1)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=1)
for batch in train_loader:
    inputs = batch['inputs']
    targets = batch['targets']
    break

In [215]:
import torch
import torch.nn as nn

class ImprovedSocialLSTM(nn.Module):
    def __init__(self, 
                 hidden_size=64,         # LSTM hidden state size
                 social_size=32,         # Social context embedding size
                 num_layers=1,           # Number of LSTM layers
                 input_frames=10,        # Number of input frames 
                 output_size=2,          # Number of outputs: [position, confidence]
                 dropout=0.2,            # Dropout probability
                 device='cuda' if torch.cuda.is_available() else 'cpu'
                 
):         # GPU acceleration
        """
        Improved Social LSTM with confidence output
        """
        super(ImprovedSocialLSTM, self).__init__()
        self.device = device
        self.hidden_size = hidden_size
        self.social_size = social_size
        self.input_frames = input_frames
        self.output_size = output_size
        
        # Position embedding
        self.position_embedding = nn.Linear(1, hidden_size)

        
        # Social context embeddings with explicit presence flags
        self.front_vehicle_embedding = nn.Linear(2, social_size)  # [distance, presence_flag]

        self.back_vehicle_embedding = nn.Linear(2, social_size)   # [distance, presence_flag]

        
        # Combine social embeddings into context
        self.social_context_combine = nn.Linear(2 * social_size, hidden_size)

        
        # LSTM for sequence processing
        self.lstm = nn.LSTM(
            input_size=2 * hidden_size,  # Position embedding + social context
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        
        # Output layer - now outputs position and confidence
        self.output_layer = nn.Linear(hidden_size, output_size)

        # Activation
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()  # For confidence scoring
        self.dropout = nn.Dropout(dropout)
        self.to(device)  # Move model to device
            

    def forward(self, inputs):
        inputs = self.prepare_input_data(inputs[:,:,0], inputs[:, :, 1], inputs[:, :, 2])
        """
        Forward pass with confidence output
        
        Args:
            inputs: Input tensor structure [batch_size, input_frames, 5]
                inputs[:,:,0] = tracked vehicle position
                inputs[:,:,1] = distance to front vehicle (0 if none)
                inputs[:,:,2] = front vehicle presence flag (0 or 1)
                inputs[:,:,3] = distance to back vehicle (0 if none)
                inputs[:,:,4] = back vehicle presence flag (0 or 1)
            
        Returns:
            outputs: Predicted [position, confidence] [batch_size, output_size]
        """
        batch_size = inputs.size(0)
        seq_length = inputs.size(1)
        
        # Prepare output tensor
        lstm_inputs = torch.zeros(batch_size, seq_length, 2 * self.hidden_size,device=self.device)
        
        # Process each time step to create embeddings
        for t in range(seq_length):
            # Extract data for current time step
            pos = inputs[:, t, 0].unsqueeze(1)  # [batch, 1]
            
            front_data = inputs[:, t, 1:3]      # [batch, 2] - [distance, presence]
            back_data = inputs[:, t, 3:5]       # [batch, 2] - [distance, presence]
            
            # Embed position
            pos_embedded = self.dropout(self.relu(self.position_embedding(pos)))
            
            # Embed social context with presence flags
            front_embedded = self.dropout(self.relu(self.front_vehicle_embedding(front_data)))
            back_embedded = self.dropout(self.relu(self.back_vehicle_embedding(back_data)))
            
            # Combine social context
            social_context = torch.cat((front_embedded, back_embedded), dim=1)
            social_embedded = self.dropout(self.relu(self.social_context_combine(social_context)))
            
            # Combine all features
            lstm_inputs[:, t] = torch.cat((pos_embedded, social_embedded), dim=1)
        
        # Process through LSTM
        lstm_out, (h_n, _) = self.lstm(lstm_inputs)
        
        # Get raw outputs
        raw_output = self.output_layer(h_n[-1])
        
        # Split into position and confidence
        position = raw_output[:, 0]
        confidence = self.sigmoid(raw_output[:, 1])  # Sigmoid to get 0-1 confidence
        
        # Combine into final output
        output = torch.cat((position.unsqueeze(1), confidence.unsqueeze(1)), dim=1)
        
        return output

    def prepare_input_data(self, positions, front_distances, back_distances):
        """
        Prepare input data with proper presence flags
        
        Args:
            positions: Positions of tracked vehicles [batch, seq_len]
            front_distances: Distances to front vehicles [batch, seq_len] (or None)
            back_distances: Distances to back vehicles [batch, seq_len] (or None)
            
        Returns:
            inputs: Formatted input tensor [batch, seq_len, 5]
        """
        batch_size = positions.size(0)
        seq_len = positions.size(1)
        
        # Create input tensor with proper structure
        inputs = torch.zeros(batch_size, seq_len, 5,device=self.device)
        
        # Set positions
        inputs[:, :, 0] = positions
        
        # Set front vehicle info with presence flags
        if front_distances is not None:
            # Where distances are valid (not None), set presence flag to 1
            front_present = (front_distances > -200).float() 
            inputs[:, :, 1] = front_distances
            inputs[:, :, 2] = front_present
        
        # Set back vehicle info with presence flags
        if back_distances is not None:
            # Where distances are valid (not None), set presence flag to 1
            back_present = (back_distances < 200).float()
            inputs[:, :, 3] = back_distances
            inputs[:, :, 4] = back_present
        
        return inputs


In [213]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\train\train_dataset.h5')
val_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\val\val_dataset.h5')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=1)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=1)
for batch in train_loader:
    inputs = batch['inputs'].to(device)
    targets = batch['targets'].to(device)
    break

In [227]:
class ConfidenceLoss(nn.Module):
    """
    Custom loss function that uses predicted confidence
    """
    def __init__(self):
        super(ConfidenceLoss, self).__init__()
        
    def forward(self, outputs, targets):
        # Extract predicted position and confidence
        pred_position = outputs[:, 0]
        pred_confidence = outputs[:, 1]
        
        # Position error (MSE)
        position_error = (pred_position - targets.squeeze()) ** 2
        
        # Scale error by confidence and add confidence regularization
        # Higher confidence → higher penalty for being wrong
        # Lower confidence → lower penalty but penalize low confidence
        confidence_penalty = -torch.log(pred_confidence)
        loss = (position_error * pred_confidence + confidence_penalty).mean()
        
        # Return loss components for logging
        return {
            'total_loss': loss,
            'position_error': position_error.mean(),
            'confidence': pred_confidence.mean()
        }

In [228]:
criterion = ConfidenceLoss().to(device)

In [226]:
model = ImprovedSocialLSTM().to(device)
model.forward(inputs)

tensor([[ 0.0562,  0.5194],
        [ 0.0458,  0.5203],
        [ 0.5661,  0.6597],
        [-0.1874,  0.4122]], device='cuda:0', grad_fn=<CatBackward0>)